# E-commerce Funnel Analysis - Exploration

## Problem Statement

E-commerce platforms typically assume a linear purchase funnel (view -> cart -> purchase) 
and design abandonment-recovery strategies around it. This analysis examines whether 
that assumption holds for this dataset, and segments user behavior across the funnel 
stages to identify the actual purchase paths users take - with the goal of informing 
where funnel optimization efforts (cart experience vs. direct-buy flow) would have 
the most impact.

### 1. Load Data & Check Columns

In [11]:
import pandas as pd

df = pd.read_csv('../data/2019-Oct.csv')
print(df.columns.tolist())

['event_time', 'event_type', 'product_id', 'category_id', 'category_code', 'brand', 'price', 'user_id', 'user_session']


### 2. Data Shape, Types & Missing Values

In [12]:
print(df.shape)
print(df.dtypes)
df.isnull().sum()

(1048575, 9)
event_time           str
event_type           str
product_id         int64
category_id      float64
category_code        str
brand                str
price            float64
user_id            int64
user_session         str
dtype: object


event_time            0
event_type            0
product_id            0
category_id           0
category_code    334761
brand            155682
price                 0
user_id               0
user_session          0
dtype: int64

### 3. Event Type Distribution

In [13]:
df['event_type'].value_counts()

event_type
view        1016239
purchase      17296
cart          15040
Name: count, dtype: int64

### 4. Funnel Overview - Naive Conversion Rates

In [14]:
total_events = df.shape[0]
view = df[df['event_type']=='view'].shape[0]
cart = df[df['event_type']=='cart'].shape[0]
purchase = df[df['event_type']=='purchase'].shape[0]

print(f"View: {view}")
print(f"Cart: {cart}")
print(f"Purchase: {purchase}")
print(f"View -> Cart rate: {cart/view*100:.2f}%")
print(f"Cart -> Purchase rate: {purchase/cart*100:.2f}%")
print(f"View -> Purchase rate: {purchase/view*100:.2f}%")

View: 1016239
Cart: 15040
Purchase: 17296
View -> Cart rate: 1.48%
Cart -> Purchase rate: 115.00%
View -> Purchase rate: 1.70%


### 5. Session-Level Check - Purchases Without Cart Events

In [15]:
# For each session, check what event types occurred
session_events = df.groupby('user_session')['event_type'].apply(set)

# Sessions that have a purchase
purchase_sessions = session_events[session_events.apply(lambda x: 'purchase' in x)]

# Of those, how many also have a cart event?
has_cart = purchase_sessions.apply(lambda x: 'cart' in x)

print(f"Total sessions with a purchase: {len(purchase_sessions)}")
print(f"Purchase sessions WITH a cart event: {has_cart.sum()}")
print(f"Purchase sessions WITHOUT a cart event: {(~has_cart).sum()}")
print(f"% of purchases with no cart in session: {(~has_cart).sum()/len(purchase_sessions)*100:.2f}%")

Total sessions with a purchase: 14693
Purchase sessions WITH a cart event: 5080
Purchase sessions WITHOUT a cart event: 9613
% of purchases with no cart in session: 65.43%


### 6. Funnel by Unique Users (Corrected)

In [16]:
unique_viewers = df[df['event_type']=='view']['user_id'].nunique()
unique_carters = df[df['event_type']=='cart']['user_id'].nunique()
unique_purchasers = df[df['event_type']=='purchase']['user_id'].nunique()

print(f"Unique users who viewed: {unique_viewers}")
print(f"Unique users who added to cart: {unique_carters}")
print(f"Unique users who purchased: {unique_purchasers}")
print(f"View -> Cart rate: {unique_carters/unique_viewers*100:.2f}%")
print(f"Cart -> Purchase rate: {unique_purchasers/unique_carters*100:.2f}%")
print(f"View -> Purchase rate: {unique_purchasers/unique_viewers*100:.2f}%")

Unique users who viewed: 168868
Unique users who added to cart: 7880
Unique users who purchased: 12774
View -> Cart rate: 4.67%
Cart -> Purchase rate: 162.11%
View -> Purchase rate: 7.56%


### 7. User Overlap Across Stages

In [17]:
viewers = set(df[df['event_type']=='view']['user_id'])
carters = set(df[df['event_type']=='cart']['user_id'])
purchasers = set(df[df['event_type']=='purchase']['user_id'])

print(f"Viewed only (never cart, never purchase): {len(viewers - carters - purchasers)}")
print(f"Viewed + Cart, no purchase: {len((viewers & carters) - purchasers)}")
print(f"Viewed + Purchase, no cart (direct buy): {len((viewers & purchasers) - carters)}")
print(f"Did all three (view + cart + purchase): {len(viewers & carters & purchasers)}")

Viewed only (never cart, never purchase): 153030
Viewed + Cart, no purchase: 3093
Viewed + Purchase, no cart (direct buy): 7963
Did all three (view + cart + purchase): 4782


### 8. Clean Data Types & Handle Missing Values

In [18]:
df['event_time'] = pd.to_datetime(df['event_time'])
df['category_code'] = df['category_code'].fillna('unknown')
df['brand'] = df['brand'].fillna('unknown')

print(df['event_time'].dtype)
print(df[['category_code','brand']].isnull().sum())

datetime64[us, UTC]
category_code    0
brand            0
dtype: int64


### Conclusion - Exploratory Findings

**Dataset:** ~1.05M events, October 2019, single month, e-commerce clickstream (view/cart/purchase).

**Key Findings:**

1. **Funnel is not sequential.** A standard view -> cart -> purchase model does not apply here. 65% of purchase sessions (9,613 of 14,693) had no cart event at all.

2. **Direct purchases dominate over cart-based purchases.** At the unique-user level, 7,963 users purchased without ever adding to cart-compared to only 4,782 who completed the full view → cart → purchase path, and 3,093 who added to cart but never bought.

3. **"Add to cart" is a weak funnel signal in this dataset.** Cart abandonment framing (common in textbook funnels) doesn't hold - most buyers skip the cart step entirely, likely via direct/"buy now" flows or session-tracking gaps.

4. **Data quality:** `category_code` was missing in 31.9% of rows, `brand` in 14.8% - both filled as "unknown" rather than dropped, to avoid losing valid view/purchase/cart records.

5. **Implication for funnel modeling:** Stages must be treated as independent counts (total unique viewers, cart-users, purchasers) rather than a strict conversion chain. Any "conversion rate" calculated as a simple sequential ratio (e.g., cart→purchase) produces invalid results (>100%) due to this non-linear behavior.

**Next step:** Export cleaned dataset for further analysis / Power BI dashboard build.

### 9. Export Cleaned Data for Power BI

In [19]:
df.to_csv('../outputs/cleaned_ecommerce_oct2019.csv', index=False)
print("Exported successfully")
print(f"File size check: {df.shape}")

Exported successfully
File size check: (1048575, 9)


### 10. Fix category_id Formatting (Scientific Notation Issue)

In [20]:
df['category_id'] = df['category_id'].astype('int64')
df.to_csv('../outputs/cleaned_ecommerce_oct2019.csv', index=False)
print(df['category_id'].head())

0    2103810000000000000
1    2053010000000000000
2    2053010000000000000
3    2053010000000000000
4    2053010000000000000
Name: category_id, dtype: int64
